<a href="https://colab.research.google.com/github/Balachandar-Ganesan/DeepLearning/blob/main/200_5_BuildYourOwnLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://raw.githubusercontent.com/Balachandar-Ganesan/DeepLearning/refs/heads/main/TinyStories-200.txt

--2026-03-06 11:03:05--  https://raw.githubusercontent.com/Balachandar-Ganesan/DeepLearning/refs/heads/main/TinyStories-200.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 23527 (23K) [text/plain]
Saving to: ‘TinyStories-200.txt’

TinyStories-200.txt 100%[===================>]  22.98K  --.-KB/s    in 0s      

2026-03-06 11:03:05 (122 MB/s) - ‘TinyStories-200.txt’ saved [23527/23527]



In [ ]:
!pip3 install tiktoken
!pip3 list | grep tiktoken

In [2]:
book_contents = ""
with open("/content/TinyStories-200.txt", "r") as f:
  book_contents = f.read()

In [4]:
import tiktoken
encoding = tiktoken.get_encoding("o200k_base")

In [5]:
encoding.encode("Timing beats Speed. Precision beats Power")




[81398, 54439, 23451, 13, 87969, 54439, 10079]

In [9]:
encoding.decode([54439])


' beats'

In [14]:
def generate_training_data(data, n, tokenizer):
    tokens = tokenizer.encode(data,disallowed_special=())
    X = []
    y = []
    for i in range(len(tokens) - n):
      X.append(tokens[i : n + i])
      y.append(tokens[i + 1 : n + i + 1])

    return [X, y]

In [15]:
sequence_len = 5
_X, _y = generate_training_data(book_contents, sequence_len, encoding)

In [17]:
import torch

tensor_X = torch.tensor(_X, dtype = torch.long)
tensor_y = torch.tensor(_y, dtype = torch.long)

#tensor_X = tensor_X.to("cuda")
#tensor_y = tensor_y.to("cuda")


In [18]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(tensor_X, tensor_y)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

In [19]:
import torch.nn as nn

In [21]:
class TinyLLM(nn.Module):
  def __init__(self, vocab_size, embed_size, hidden_size):
    super(TinyLLM, self).__init__()
    self.embedding = nn.Embedding(vocab_size, embed_size)
    self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)
    self.fc = nn.Linear(hidden_size, vocab_size)

  def forward(self, x):
    out = self.embedding(x)
    out, _ = self.rnn(out)
    out = self.fc(out)
    return out


#TinyLLM.forward = forward

In [23]:
embed_size = 128
hidden_size = 256

model = TinyLLM(encoding.n_vocab,
                embed_size, hidden_size
)

In [24]:
num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params}")

Total parameters: 77106131


In [25]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [27]:
n_epochs = 1

for epoch in range(n_epochs):
  epoch_loss = 0
  for X, y in dataloader:
    optimizer.zero_grad()
    outputs = model(X)
    outputs = outputs.view(-1, encoding.n_vocab)
    y = y.view(-1)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    epoch_loss += loss.item()

  avg_loss = epoch_loss / len(dataloader)
  print(f'Epoch [{epoch+1}/{n_epochs}], Loss: {avg_loss:.4f}')

Epoch [1/1], Loss: 5.7164


In [28]:
torch.save(model.state_dict(), "tinyllm40_weights.pth")


In [29]:
model.eval()

TinyLLM(
  (embedding): Embedding(200019, 128)
  (rnn): RNN(128, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=200019, bias=True)
)

In [33]:
import torch.nn.functional as F

def generate_text(prompt, tokenizer, max_length = 50):
  prompt = tokenizer.encode(prompt)[-sequence_len:]
  generated = prompt.copy()
  with torch.no_grad():
    for _ in range(max_length):
      current = [generated[-sequence_len:]]
      current = torch.tensor(current, dtype=torch.long)
      output = model(current)
      predictions = output[:, -1, :]
      probabilities = F.softmax(predictions, dim=-1)
      next_token = torch.multinomial(probabilities, num_samples=1).item()
      generated.append(next_token)
  print(encoding.decode(generated))

In [34]:
generate_text("honest police officer", encoding, max_length=25)


honest police officer that up leading he a their legs Indian < <|endoftext|>
A happening political her and|endoftext


In [35]:
generate_text("best friends become worst enemies", encoding, max_length=25)


best friends become worst enemies|endoftext|>
A group- theofof local foreign>
AThree that Muse caring experiences <|endof
